In [1]:
#setup
import pandas as pd
import numpy as np
import duckdb
from google.colab import drive
drive.mount("/content/drive")

db_path = "/content/drive/MyDrive/mapping_mortality_gap/data_clean/staging.duckdb"


Mounted at /content/drive


In [2]:
#combining all the table years into one dataframe with cause Cardiomyopathy (I42)
conn = duckdb.connect(db_path)

extract_query = ""
for i in range(2018, 2025):
  extract_query += f"SELECT * FROM mortality_{i} WHERE cause LIKE 'I42%'"
  if i != 2024:
    extract_query += "\nUNION ALL\n"

df = conn.execute(extract_query).df()
conn.close()

print(df.head())


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   year month sex   age race cause
0  2018    01   F  1073   03  I429
1  2018    01   F  1060   01  I425
2  2018    02   M  1075   03  I429
3  2018    01   M  1030   01  I420
4  2018    04   M  1049   03  I426


In [3]:
#relabeling sex to Male & Female
sex_map = {
    'F': 'Female',
    'M': 'Male'
}

df["sex"] = df["sex"].map(sex_map)
print(df.head())

   year month     sex   age race cause
0  2018    01  Female  1073   03  I429
1  2018    01  Female  1060   01  I425
2  2018    02    Male  1075   03  I429
3  2018    01    Male  1030   01  I420
4  2018    04    Male  1049   03  I426


In [4]:
#Standardize Cause of Death (new column)
df["cause_of_death"] = "Cardiomyopathy (I42)"

#Relabelling the race
race_conditions= [
    (df['race'] == '01'),
    (df['race'] == '02'),
    (df['race'] == '03'),
    (df['race'] == '04'),
    (df['race'] == '05'),
    (df['race'] == '06'),
    (df['race'] == '07'),
    (df['race'] == '08'),
    (df['race'] == '09'),
    (df['race'] == '10'),
    (df['race'] == '11'),
    (df['race'] == '12'),
    (df['race'] == '13'),
    (df['race'] == '14'),
]

race_choices = [
    'White',
    'Black',
    'American Indian and Alaskan Native',
    'Asian Indian',
    'Chinese',
    'Filipino',
    'Japanese',
    'Korean',
    'Vietnamese',
    'Other Asian',
    'Hawaiian',
    'Guamanian or Chamorro',
    'Samoan',
    'Other Pacific Islander',
]

df["race_category"] = np.select(race_conditions, race_choices, default="Others")


# reballing the months
month_condition = [
    (df['month'] == "01"),
    (df['month'] == "02"),
    (df['month'] == "03"),
    (df['month'] == "04"),
    (df['month'] == "05"),
    (df['month'] == "06"),
    (df['month'] == "07"),
    (df['month'] == "08"),
    (df['month'] == "09"),
    (df['month'] == "10"),
    (df['month'] == "11"),
    (df['month'] == "12"),
]

month_choice = [
    "January",
    "February",
    "March",
    "April",
    "May",
    "June",
    "July",
    "August",
    "September",
    "October",
    "November",
    "December"
]
df["months"] = np.select(month_condition, month_choice, default="Out of range")


df.head()

,year,month,sex,age,race,cause,cause_of_death,race_category,months
0,2018,01,Female,1073,03,I429,Cardiomyopathy (I42),American Indian and Alaskan Native,January
1,2018,01,Female,1060,01,I425,Cardiomyopathy (I42),White,January
2,2018,02,Male,1075,03,I429,Cardiomyopathy (I42),American Indian and Alaskan Native,February
3,2018,01,Male,1030,01,I420,Cardiomyopathy (I42),White,January
4,2018,04,Male,1049,03,I426,Cardiomyopathy (I42),American Indian and Alaskan Native,April


In [5]:
# Relabel the age column (4 digits).
# The first digit indicates the age unit: 1 = Years, 2 = Months, 4 = Days, 9 = Unknown.
# The remaining three digits represent the age value. Examples:
# 1042 → 42 years (Person died at age of 42 years because of Cardiomyopathy )
# 2042 → 42 months (Person died at after 42 months of birth because of Cardiomyopathy )

df["age_numeric"] = pd.to_numeric(df["age"].str[1:], errors="coerce")

age_condition = [
    (df["age"].str.startswith("1")) & (df["age_numeric"] <= 24),
    (df['age'].str.startswith("1")) & (df['age_numeric'] >=25) & (df['age_numeric'] <= 44),
    (df['age'].str.startswith("1")) & (df['age_numeric'] >=45) & (df['age_numeric']<=64),
    (df["age"].str.startswith("1")) & (df["age_numeric"] >=65)
]

age_choices = [
    "0-24",
    "25-44",
    "45-64",
    "65+"
]

df["age_group"] = np.select(age_condition, age_choices, default="Exclude")

#removing columns with exclude Age value
final_df = df[df["age_group"] != "Exclude"].copy()
df.head()

,year,month,sex,age,race,cause,cause_of_death,race_category,months,age_numeric,age_group
0,2018,01,Female,1073,03,I429,Cardiomyopathy (I42),American Indian and Alaskan Native,January,73,65+
1,2018,01,Female,1060,01,I425,Cardiomyopathy (I42),White,January,60,45-64
2,2018,02,Male,1075,03,I429,Cardiomyopathy (I42),American Indian and Alaskan Native,February,75,65+
3,2018,01,Male,1030,01,I420,Cardiomyopathy (I42),White,January,30,25-44
4,2018,04,Male,1049,03,I426,Cardiomyopathy (I42),American Indian and Alaskan Native,April,49,45-64


In [6]:
# dropping the temportary helper columns and unncessary columns
final_df = df.drop(columns=['age_numeric', 'age', 'race', 'cause', 'month'])

#reordering the month column after year
months = final_df.pop("months")
final_df.insert(1, "months", months)

final_df

,year,months,sex,cause_of_death,race_category,age_group
0,2018,January,Female,Cardiomyopathy (I42),American Indian and Alaskan Native,65+
1,2018,January,Female,Cardiomyopathy (I42),White,45-64
2,2018,February,Male,Cardiomyopathy (I42),American Indian and Alaskan Native,65+
3,2018,January,Male,Cardiomyopathy (I42),White,25-44
4,2018,April,Male,Cardiomyopathy (I42),American Indian and Alaskan Native,45-64
...,...,...,...,...,...,...
139805,2024,December,Male,Cardiomyopathy (I42),White,45-64
139806,2024,December,Male,Cardiomyopathy (I42),Chinese,0-24
139807,2024,December,Male,Cardiomyopathy (I42),Black,65+
139808,2024,December,Male,Cardiomyopathy (I42),White,25-44


In [8]:
#Export the data
final_df.to_csv("/content/drive/MyDrive/mapping_mortality_gap/data_clean/national_microdata_clean.csv", index=False)